## Imports

In [ ]:
from pathlib import Path
import os
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

## Load damage data

In [ ]:
data_folder_path = Path('./data')
# damagedata = data_folder_path / "Florida_fixed_building_sample_florida.csv"
damagedata = data_folder_path / "Florida_fixed_building_sample_florida_10m_v2.csv"

df_nbs = pd.read_csv(damagedata)

# Clean damages column (remove $, commas, spaces)
df = df_nbs.copy()

currency_cols = [
  'mean_damage',
  'std_damage',
  'mean_building_value',
  'std_building_value'
]

for col in currency_cols:
    if col in df.columns:
        # Remove $, commas, and spaces, then convert to float
        df[col] = (df[col]
                   .astype(str)              # Ensure it's string first
                   .str.strip()              # Remove leading/trailing spaces
                   .str.replace('$', '', regex=False)
                   .str.replace(',', '', regex=False)
                   .astype(float))

df['rp_clean'] = df['return_period'].str.replace('RP', '', regex=False)
df['rp_clean'] = pd.to_numeric(df['rp_clean'])

df.head()

## Load number of total buildings subject to flooding

In [ ]:
###import number of total buildings subject to flooding
df_total = pd.read_csv(data_folder_path / "Florida_damage_dollar_stats_by_county_florida_10m_v2.csv")

for col in [
    'mean_damage_dollar',
    'std_damage_dollar',
    'p10_damage_dollar',
    'p25_damage_dollar',
    'p75_damage_dollar',
    'p90_damage_dollar'
    ]:
    df_total[col] = (df_total[col].astype(str).str.strip()
                    .str.replace('$', '', regex=False)
                    .str.replace(',', '', regex=False)
                    .astype(float))

df_total.head()

## Load import total number of buildings + aggregate value

In [ ]:
###import total number of buildings + aggregate value
df_all_damages = pd.read_csv(data_folder_path / "ALL_damages_by_county_florida_10m_v2.csv")

# Clean currency columns
for col in ['val_struct_flooded', 'val_cont_flooded', 'total_damage']:
    df_all_damages[col] = df_all_damages[col].str.replace('$', '').str.replace(',', '').str.strip().astype(float)

df_all_damages.head()

## Output folder

In [ ]:
# Define output folder
output_folder = Path('./output')

# Create the directory if it doesn't exist
output_folder.mkdir(parents=True, exist_ok=True)

print(f"✓ Output folder ready: {output_folder}")

## Configuration

In [ ]:
# ============================================================================
# SECTION 1: PARAMETERS
# ============================================================================

# --- Housing Parameters 
DELTA = 0.07               # Rental yield 

# --- SDF Parameters 
BETA = 0.95                  # Discount factor 
GAMMA = 2                    # Risk aversion coefficient
G_C = 0.025                  # Consumption growth rate (2.5%)

XI_C_MAP = {10: 0.01, 25: 0.02, 50: 0.04, 100: 0.08}

COUNTIES = [
    'Charlotte',
    'Collier',
    'Hillsborough',
    'Lee',
    'Manatee',
    'Miami-Dade',
    'Pinellas'
]

# Group parameters into a dictionary for easier passing
PARAMS = {
    'delta': DELTA,
    'beta': BETA,
    'gamma': GAMMA,
    'gc': G_C,
    'xi_c_map': XI_C_MAP
}

In [ ]:
###the following simulation determines the TIMING Of disasters
# Parameters
N_SIMULATIONS = 1000
YEAR_START = 2024
YEAR_END = 2100
T = YEAR_END - YEAR_START + 1

# Discrete probabilities (not ranges)
storm_probs = {
    '10yr': 0.10,   # 10% annual probability
    '25yr': 0.04,   # 4% annual probability
    '50yr': 0.02,   # 2% annual probability
    '100yr': 0.01   # 1% annual probability
}

storm_probs_climate = {
    '10yr': 0.10,   # CC modeled via depth/extent; storm frequency held at baseline
    '25yr': 0.04,
    '50yr': 0.02,
    '100yr': 0.01
}

## Storm-matrix simulation

In [ ]:
np.random.seed(42)

def generate_storm_matrix_independent(N_sim, T, probs, climate_scenario='baseline'):
    """
    Generate storm matrix with independent Bernoulli draws

    Returns:
    --------
    S_matrix : ndarray (N_sim, T, 4)
        Binary matrix: 1 if storm type occurred, 0 otherwise
        Dimension 3: [10yr, 25yr, 50yr, 100yr]
    """

    # Initialize 3D matrix: paths × years × storm types
    S_matrix = np.zeros((N_sim, T, 4), dtype = int)

    # Get probabilities
    if climate_scenario == 'baseline':
        p = [probs['10yr'], probs['25yr'], probs['50yr'], probs['100yr']]
    else:
        # For climate change, will modify later if assuming time-varying probabilities
        p = [probs['10yr'], probs['25yr'], probs['50yr'], probs['100yr']]

    # Draw independently for each storm type
    for storm_idx in range(4):
        S_matrix[:, :, storm_idx] = np.random.binomial(1, p[storm_idx], size=(N_sim, T))

    return S_matrix

# Generate baseline storm matrix
S_matrix_baseline = generate_storm_matrix_independent(
    N_SIMULATIONS,
    T,
    storm_probs,
    climate_scenario='baseline'
)

print("✓ Storm matrix generated (independent Bernoulli)")
print(f"  Shape: {S_matrix_baseline.shape} (paths × years × storm types)")

# Quick check
total_10yr = S_matrix_baseline[:, :, 0].sum()
total_100yr = S_matrix_baseline[:, :, 3].sum()
print(f"  10-year storms: {total_10yr} (expected ~{N_SIMULATIONS*T*0.10:.0f})")
print(f"  100-year storms: {total_100yr} (expected ~{N_SIMULATIONS*T*0.01:.0f})")

## Helper: damage-ratio lookup

In [ ]:
def get_damage_ratio(county, return_period, scenario, df, stat_type='mean', use_probability=False, df_total=None):
    """
    Get damage ratio ξ_h directly from the dataframe, with optional probability adjustment.

    Parameters:
    -----------
    df : DataFrame
        The new df_v3 dataframe (or equivalent baseline).
    stat_type : str
        'mean' to determine which ratio to fetch (kept generic for legacy reasons).
    use_probability : bool
        If True, apply probability reduction for S1/S1CC scenarios.
    df_total : DataFrame
        Required if use_probability=True (Florida_damage_dollar_stats_by_county_florida.csv).

    Returns:
    --------
    float : Damage ratio (e.g., 0.02 for 2% damage)
    """
    rp_string = f'RP{return_period:03d}'

    row = df[(df['county'] == county) &
                  (df['scenario'] == scenario) &
                  (df['return_period'] == rp_string)]

    if len(row) == 0:
        return 0.0

    # Fetch the {stat_type}_damage_ratio column (paper uses 'mean')
    ratio_col = f'{stat_type}_damage_ratio'

    if ratio_col not in row.columns:
        raise ValueError(f"Column {ratio_col} not found in dataframe.")

    base_ratio = float(row.iloc[0][ratio_col])

    # Apply probability adjustment if requested
    if use_probability and scenario in ['S1', 'S1CC'] and df_total is not None:
        # Determine the corresponding 'without mangroves' scenario for probability adjustment
        if scenario == 'S1':
            scenario_for_n_with = 'S1'
            scenario_for_n_without = 'S2'
        elif scenario == 'S1CC':
            scenario_for_n_with = 'S1CC'
            scenario_for_n_without = 'S2CC'
        else:
            # This case should ideally not be reached if scenario is in ['S1', 'S1CC']
            return base_ratio # No probability adjustment if conditions are not met

        # Get building counts for probability ratio
        s_with_row = df_total[(df_total['county'] == county) &
                         (df_total['scenario'] == scenario_for_n_with) &
                         (df_total['return_period'] == rp_string)]

        s_without_row = df_total[(df_total['county'] == county) &
                         (df_total['scenario'] == scenario_for_n_without) &
                         (df_total['return_period'] == rp_string)]

        if len(s_with_row) > 0 and len(s_without_row) > 0:
            n_with = s_with_row.iloc[0]['n_buildings_flooded']
            n_without = s_without_row.iloc[0]['n_buildings_flooded']

            # Avoid division by zero
            if n_without == 0:
                return min(base_ratio, 1.0)

            # Probability of flooding given storm occurs
            p_flood = n_with / n_without

            # Adjust damage by probability
            adjusted_ratio = base_ratio * p_flood

            return min(adjusted_ratio, 1.0)

    return min(base_ratio, 1.0)

## Sanity check: damage-ratio lookup

In [ ]:
def check_get_damage_ratio(df, df_total):
    print("--- Testing get_damage_ratio ---")

    test_cases = [
        {'county': 'Charlotte', 'rp': 100, 'scenario': 'S1', 'use_prob': False, 'stat': 'mean'},
        {'county': 'Charlotte', 'rp': 100, 'scenario': 'S1', 'use_prob': True, 'stat': 'mean'},
        {'county': 'Collier', 'rp': 10, 'scenario': 'S1CC', 'use_prob': True, 'stat': 'mean'}
    ]

    for tc in test_cases:
        try:
            result = get_damage_ratio(
                county=tc['county'],
                return_period=tc['rp'],
                scenario=tc['scenario'],
                df=df,
                stat_type=tc['stat'],
                use_probability=tc['use_prob'],
                df_total=df_total
            )
            print(f"County: {tc['county']:<12} | RP: {tc['rp']:<3} | Scenario: {tc['scenario']:<4} | Stat: {tc['stat']:<6} | Use Prob: {str(tc['use_prob']):<5} => Result: {result:.6f}")
        except Exception as e:
            print(f"Error testing {tc}: {e}")

check_get_damage_ratio(df, df_total)

## Run engineering simulations across counties

In [ ]:
# ============================================================================
# EXACT IMPLEMENTATION OF EQUATION 1 & 3
# ============================================================================

def calculate_H0_equation1(S_path, county, scenario, df, df_total, stat_type='mean', use_probability=False, params=PARAMS):
    """
    Implement Equation 1 EXACTLY as written:

    H_0 = E[∑(j=1 to T) (1+δ)^j × (∏(k=1 to j) m_k(1-s_k ξ_h))] × (1-s_0 ξ_h)

    No V_0, no additional scaling - just the equation as written.

    Returns:
    - H_0 from Equation 1 (in consumption units)
    """

    delta = params['delta']
    beta = params['beta']
    gamma = params['gamma']
    gc = params['gc']

    T = S_path.shape[0]
    storm_rp_map = {0: 10, 1: 25, 2: 50, 3: 100}

    # ========================================================================
    # STEP 1: SIMULATE CONSUMPTION PATH (for m_k)
    # ========================================================================

    C = np.zeros(T)
    C[0] = 1.0

    for t in range(T):
        if t > 0:
            C[t] = C[t-1] * (1 + gc)

        # Apply consumption shock if storm occurs
        storms_this_year = []
        for storm_idx in range(4):
            if S_path[t, storm_idx] == 1:
                storms_this_year.append(storm_rp_map[storm_idx])

        if len(storms_this_year) > 0:
            largest_rp = max(storms_this_year)
            xi_c = XI_C_MAP[largest_rp]
            C[t] *= (1 - xi_c)

    # ========================================================================
    # STEP 2: GET DAMAGE RATIOS FOR EACH PERIOD
    # ========================================================================

    xi_h = np.zeros(T)

    for t in range(T):
        storms_this_year = []
        for storm_idx in range(4):
            if S_path[t, storm_idx] == 1:
                storms_this_year.append(storm_rp_map[storm_idx])

        if len(storms_this_year) > 0:
            largest_rp = max(storms_this_year)
            xi_h[t] = get_damage_ratio(county, largest_rp, scenario, df, use_probability=use_probability, df_total = df_total, stat_type=stat_type)

    # ========================================================================
    # STEP 3: IMPLEMENT EQUATION 1 EXACTLY
    # ========================================================================

    # Calculate: ∑(j=1 to T) (1+δ)^j × (∏(k=1 to j) m_k(1-s_k ξ_h))

    total_sum = 0.0

    for j in range(1, T):  # j from 1 to T-1
        # Calculate (1+δ)^j
        service_term = (1 + delta) ** j

        # Calculate ∏(k=1 to j) m_k × (1-s_k ξ_h)
        cumulative_product = 1.0

        for k in range(1, j+1):  # k from 1 to j (using 1-indexed as in equation)
            # m_k = β × (C[k]/C[k-1])^(-γ)
            m_k = beta * ((C[k] / C[k-1]) ** (-gamma))

            # Damage factor at time k
            damage_factor_k = (1 - xi_h[k])

            # Multiply into cumulative product
            cumulative_product *= m_k * damage_factor_k

        # Add this term to sum
        total_sum += service_term * cumulative_product

    # Multiply by (1-s_0 ξ_h) - damage at time 0
    H_0 = total_sum * (1 - xi_h[0])

    return H_0

# ============================================================================
# MAIN CALCULATION FUNCTION
# ============================================================================

def calculate_one_path_all_storms(S_path, county, scenario_label, df, df_total, stat_type='mean',
                                  use_mangroves=False, use_probability=False, params=PARAMS):
    """Calculate H_0 for one simulation path using exact Equation 1"""

    # Map to data scenario
    if scenario_label == 'baseline':
        scenario = 'S1' if use_mangroves else 'S2'
    elif scenario_label == 'climate_change':
        scenario = 'S1CC' if use_mangroves else 'S2CC'
    else:
        raise ValueError(f"Unknown scenario_label: {scenario_label}")

    # Calculate H_0 using exact Equation 1
    H_0 = calculate_H0_equation1(S_path, county, scenario, df, df_total,
                                 use_probability=use_probability, stat_type=stat_type, params=params)

    return H_0

# ============================================================================
# RUN SIMULATIONS
# ============================================================================

def run_county_simulations(county, scenario_label, S_matrix, df, df_total, use_probability=False, stat_type='mean', params=PARAMS):
    """Run all simulations for one county/scenario"""

    N_sim = S_matrix.shape[0]
    H_0_without_list = []
    H_0_with_list = []

    print(f"  Processing {county}, {scenario_label}...")

    for i in range(N_sim):
        S_path = S_matrix[i, :, :]

        H_0_without = calculate_one_path_all_storms(
            S_path, county, scenario_label, df, df_total, stat_type=stat_type,
            use_mangroves=False, use_probability=use_probability, params=params
        )

        H_0_with = calculate_one_path_all_storms(
            S_path, county, scenario_label, df, df_total, stat_type=stat_type,
            use_mangroves=True, use_probability=use_probability, params=params
        )

        H_0_without_list.append(H_0_without)
        H_0_with_list.append(H_0_with)

        if (i + 1) % 100 == 0:
            print(f"    Completed {i + 1}/{N_sim}")

    H_0_without = np.array(H_0_without_list)
    H_0_with = np.array(H_0_with_list)

    # Calculate means first
    H_0_without_mean = np.mean(H_0_without)
    H_0_with_mean = np.mean(H_0_with)

    # Calculate protection value and percentage on the MEANS, not per path
    protection_value_mean = H_0_with_mean - H_0_without_mean

    if H_0_without_mean > 0:
        protection_pct_mean = (protection_value_mean / H_0_without_mean) * 100
    else:
        protection_pct_mean = 0.0

    results = {
        'H_0_without_mean': H_0_without_mean,
        'H_0_with_mean': H_0_with_mean,
        'H_0_without_std': np.std(H_0_without),
        'H_0_with_std': np.std(H_0_with),
        'protection_value_mean': protection_value_mean,
        'protection_pct_mean': protection_pct_mean,
    }

    print(f"    H_0(without)={results['H_0_without_mean']:.4f}, "
          f"H_0(with)={results['H_0_with_mean']:.4f}, "
          f"Protection={results['protection_value_mean']:.4f} "
          f"({results['protection_pct_mean']:.2f}%)")

    return results

In [ ]:
# ============================================================================
# RUN COMPLETE ANALYSIS
# ============================================================================

print("\n" + "="*70)
print("EXACT EQUATION 1 IMPLEMENTATION")
print("="*70)

all_results_mean = []

STAT_TYPES = ['mean']

for stat_type in STAT_TYPES:
    print(f"\n{'='*70}")
    print(f"RUNNING SIMULATIONS FOR STAT_TYPE: {stat_type.upper()}")
    print(f"{'='*70}")

    for county in COUNTIES:

        print(f"\n{'='*70}")
        print(f"COUNTY: {county}")
        print(f"{'='*70}")

        # Baseline
        results_baseline = run_county_simulations(
            county=county,
            scenario_label='baseline',
            S_matrix=S_matrix_baseline,
            df=df,
            df_total=df_total,
            stat_type=stat_type,
            use_probability=True,
            params=PARAMS
        )

        all_results_mean.append({
            'county': county,
            'scenario': 'baseline',
            **results_baseline
        })

        # Climate change
        print(f"\n  Generating climate change storm matrix...")

        S_matrix_climate = generate_storm_matrix_independent(
            N_SIMULATIONS, T, storm_probs_climate, climate_scenario='climate_change'
        )

        results_climate = run_county_simulations(
            county=county,
            scenario_label='climate_change',
            S_matrix=S_matrix_climate,
            df=df,
            df_total=df_total,
            stat_type=stat_type,
            use_probability=True,
            params=PARAMS
        )

        all_results_mean.append({
            'county': county,
            'scenario': 'climate_change',
            **results_climate
        })

# Create DataFrame for mean results
df_results_mean = pd.DataFrame(all_results_mean)

print("\n" + "="*70)
print("RESULTS - EXACT EQUATION 1 (MEAN DAMAGE RATIO)")
print("="*70)
print(df_results_mean.to_string(index=False))

# Save to output folder using the defined output_folder variable
output_path_results_mean = output_folder / 'engineering_npv_equation1_mean.csv'
df_results_mean.to_csv(output_path_results_mean, index=False)
print(f"\n✓ Saved mean results to: {output_path_results_mean}")

# Assign df_results to df_results_mean for backward compatibility with subsequent cells
df_results = df_results_mean

## Storm timeline data (Fig 2 input)

In [ ]:
# ============================================================================
# DATA FOR CHART 1: TIMELINE - Average Number of Storms by Year and Return Period
# ============================================================================

# Calculate mean number of storms per year for each return period
years = np.arange(2024, 2101)  # 77 years
N_sim = S_matrix_baseline.shape[0]

# For baseline scenario
storm_counts_baseline = np.zeros((77, 4))  # 77 years × 4 return periods

for year_idx in range(77):
    for storm_idx in range(4):
        # Count how many simulations have this storm in this year
        count = np.sum(S_matrix_baseline[:, year_idx, storm_idx])
        # Mean = count / N_sim
        storm_counts_baseline[year_idx, storm_idx] = count / N_sim

# For climate scenario
storm_counts_climate = np.zeros((77, 4))

for year_idx in range(77):
    for storm_idx in range(4):
        count = np.sum(S_matrix_climate[:, year_idx, storm_idx])
        storm_counts_climate[year_idx, storm_idx] = count / N_sim

# # Create the plot
# fig, axes = plt.subplots(2, 1, figsize=(16, 10))

# for ax_idx, (scenario_name, data) in enumerate(scenarios_data):
#     ax = axes[ax_idx]

#     # Add subtle background
#     ax.set_facecolor('#f8f9fa')

# print("✓ Chart 1 saved: storm_timeline_2024_2100.png")

In [ ]:
print("\n" + "="*70)
print("="*70)

# Create a comprehensive dataframe with all the storm data
storm_data_list = []

# Years
years = np.arange(2024, 2101)

# Return period labels
rp_labels = ['RP10', 'RP25', 'RP50', 'RP100']
rp_names = ['10-year', '25-year', '50-year', '100-year']

# Baseline scenario
for year_idx, year in enumerate(years):
    for storm_idx, (rp_label, rp_name) in enumerate(zip(rp_labels, rp_names)):
        storm_data_list.append({
            'year': year,
            'scenario': 'baseline',
            'return_period': rp_label,
            'return_period_name': rp_name,
            'mean_num_storms': storm_counts_baseline[year_idx, storm_idx],
            'probability': [0.10, 0.04, 0.02, 0.01][storm_idx]
        })

# Climate change scenario
for year_idx, year in enumerate(years):
    for storm_idx, (rp_label, rp_name) in enumerate(zip(rp_labels, rp_names)):
        storm_data_list.append({
            'year': year,
            'scenario': 'climate_change',
            'return_period': rp_label,
            'return_period_name': rp_name,
            'mean_num_storms': storm_counts_climate[year_idx, storm_idx],
            'probability': [0.10, 0.04, 0.02, 0.01][storm_idx]
        })

# Convert to DataFrame
df_storms = pd.DataFrame(storm_data_list)

# Save to output folder
output_path = output_folder / 'storm_timeline_data.csv'
df_storms.to_csv(output_path, index=False)

print(f"✓ Saved: {output_path}")
print(f"  - {len(df_storms)} rows (77 years × 4 return periods × 2 scenarios)")
print(f"  - {len(df_storms.columns)} columns")
print()

# Show structure
print("Columns:")
for col in df_storms.columns:
    print(f"  - {col}")
print()

print("Sample data:")
print(df_storms.head(8))
print("...")
print(df_storms.tail(8))

print("\n" + "="*70)
print("="*70)

## Hedonic damage helpers

In [ ]:
# ============================================================================
# MARKET/HEDONIC ASSESSMENT
# ============================================================================

# STEP 1: DEFINE HEDONIC DAMAGE CONFIGURATIONS
HEDONIC_CONFIGS = {
    'figure5': {
        'name': 'Figure 5',
        'without_mangroves': 0.073902,   # 19% damage at >16km
        'with_mangroves': 0.0147804,      # 2% damage at <2km
    },
    'custom': {
        'name': 'Custom',
        'without_mangroves': 0.10,
        'with_mangroves': 0.02,
    },
}

# SELECT WHICH CONFIG TO USE (change this to test different values)
ACTIVE_CONFIG = 'figure5'  # <-- CHANGE THIS TO TEST OTHERS
# ACTIVE_CONFIG = 'custom'

print(f"Using hedonic config: {HEDONIC_CONFIGS[ACTIVE_CONFIG]['name']}")
print(f"  Without mangroves: {HEDONIC_CONFIGS[ACTIVE_CONFIG]['without_mangroves']} ({HEDONIC_CONFIGS[ACTIVE_CONFIG]['without_mangroves']*100}%)")
print(f"  With mangroves: {HEDONIC_CONFIGS[ACTIVE_CONFIG]['with_mangroves']} ({HEDONIC_CONFIGS[ACTIVE_CONFIG]['with_mangroves']*100}%)")
print(f"  Differential: {(HEDONIC_CONFIGS[ACTIVE_CONFIG]['without_mangroves'] - HEDONIC_CONFIGS[ACTIVE_CONFIG]['with_mangroves'])*100} percentage points")

In [ ]:
# County-specific hedonic damages
# Pooled statewide estimate (Liu et al. 2026) for Charlotte, Collier, Lee;
# county-specific estimates for Hillsborough, Manatee, Miami-Dade, Pinellas.
hedonic_damages = {
    'Charlotte':    {'without_mangroves': 0.073902,   'with_mangroves': 0.0147804},
    'Collier':      {'without_mangroves': 0.073902,   'with_mangroves': 0.0147804},
    'Hillsborough': {'without_mangroves': 0.0950027,  'with_mangroves': 0.01900054},
    'Lee':          {'without_mangroves': 0.073902,   'with_mangroves': 0.0147804},
    'Manatee':      {'without_mangroves': 0.0486953,  'with_mangroves': 0.00973906},
    'Miami-Dade':   {'without_mangroves': 0.0493886,  'with_mangroves': 0.00987772},
    'Pinellas':     {'without_mangroves': 0.21321405, 'with_mangroves': 0.04264281},
}

for c in COUNTIES:
    d = hedonic_damages[c]
    print(f"{c:14s} without={d['without_mangroves']:.6f}  with={d['with_mangroves']:.6f}")


## Hedonic damage-ratio lookup

In [ ]:
def get_hedonic_damage_ratio(county, return_period, scenario, hedonic_data,
                             df_engineering, df_total):
    """
    Linear scaling proportional to return period.

    Assumption: RP10 causes 10% of RP100 damage, RP25 causes 25%, etc.
    This is conservative and matches the probability-damage tradeoff.
    """
    if county not in hedonic_data:
        return 0.0

    has_mangroves = scenario in ['S1', 'S1CC']

    if has_mangroves:
        xi_h_100 = hedonic_data[county]['with_mangroves']
    else:
        xi_h_100 = hedonic_data[county]['without_mangroves']

    # Linear scaling proportional to return period
    scaling = {
        10: 0.10,   # RP10 = 10% of RP100
        25: 0.25,   # RP25 = 25% of RP100
        50: 0.50,   # RP50 = 50% of RP100
        100: 1.00   # RP100 = reference
    }

    return xi_h_100 * scaling.get(return_period, 1.0)

In [ ]:
# ============================================================================
# MARKET NPV CALCULATION FUNCTION
# ============================================================================

def calculate_H0_market(S_path, county, scenario, hedonic_data,
                       df_engineering, df_total, params=PARAMS):
    """
    Calculate H_0 using HEDONIC damages (same Equation 1, different ξ_h)

    This is identical to calculate_H0_equation1 except:
    - Uses get_hedonic_damage_ratio() instead of get_damage_ratio()
    - No probability adjustment (market already priced it in)
    """

    delta = params['delta']
    beta = params['beta']
    gamma = params['gamma']
    gc = params['gc']

    T = S_path.shape[0]
    storm_rp_map = {0: 10, 1: 25, 2: 50, 3: 100}

    C = np.zeros(T)
    C[0] = 1.0

    for t in range(T):
        if t > 0:
            C[t] = C[t-1] * (1 + gc)

        # Apply consumption shock if storm occurs
        storms_this_year = []
        for storm_idx in range(4):
            if S_path[t, storm_idx] == 1:
                storms_this_year.append(storm_rp_map[storm_idx])

        if len(storms_this_year) > 0:
            largest_rp = max(storms_this_year)
            xi_c = XI_C_MAP[largest_rp]
            C[t] *= (1 - xi_c)

    xi_h = np.zeros(T)

    for t in range(T):
        storms_this_year = []
        for storm_idx in range(4):
            if S_path[t, storm_idx] == 1:
                storms_this_year.append(storm_rp_map[storm_idx])

        if len(storms_this_year) > 0:
            largest_rp = max(storms_this_year)
            # Use hedonic damage ratio (with engineering scaling)
            xi_h[t] = get_hedonic_damage_ratio(
                county, largest_rp, scenario, hedonic_data,
                df_engineering, df_total
            )

    total_sum = 0.0

    for j in range(1, T):
        service_term = (1 + delta) ** j
        cumulative_product = 1.0

        for k in range(1, j+1):
            m_k = beta * ((C[k] / C[k-1]) ** (-gamma))
            damage_factor_k = (1 - xi_h[k])
            cumulative_product *= m_k * damage_factor_k

        total_sum += service_term * cumulative_product

    H_0 = total_sum * (1 - xi_h[0])

    return H_0

In [ ]:
# ============================================================================
# RUN MARKET/HEDONIC ASSESSMENT
# ============================================================================

print("\n" + "="*70)
print("RUNNING MARKET/HEDONIC ASSESSMENT")
# print(f"Using: {HEDONIC_CONFIGS[ACTIVE_CONFIG]['name']}")  # legacy label; county-specific hedonic_damages used
print("="*70)

results_market = []

for county in COUNTIES:
    # Market/hedonic uses present-day (baseline) only; the hedonic response is present-day and market-CC is not reported
    for scenario_label in ['baseline']:

        # Get the appropriate storm matrix
        S_matrix = S_matrix_baseline if scenario_label == 'baseline' else S_matrix_climate
        N_sim = S_matrix.shape[0]

        H_0_without_list = []
        H_0_with_list = []

        print(f"\n  Processing {county}, {scenario_label} (MARKET)...")

        for i in range(N_sim):
            S_path = S_matrix[i, :, :]

            # WITHOUT mangroves
            scenario = 'S2' if scenario_label == 'baseline' else 'S2CC'
            H_0_without = calculate_H0_market(
                S_path, county, scenario, hedonic_damages,
                df, df_total, params=PARAMS
            )
            H_0_without_list.append(H_0_without)

            # WITH mangroves
            scenario = 'S1' if scenario_label == 'baseline' else 'S1CC'
            H_0_with = calculate_H0_market(
                S_path, county, scenario, hedonic_damages,
                df, df_total, params=PARAMS
            )
            H_0_with_list.append(H_0_with)

            if (i + 1) % 100 == 0:
                print(f"    Completed {i + 1}/{N_sim}")

        # Calculate statistics
        H_0_without_mean = np.mean(H_0_without_list)
        H_0_with_mean = np.mean(H_0_with_list)
        protection_value_mean = H_0_with_mean - H_0_without_mean

        if H_0_without_mean > 0:
            protection_pct_mean = (protection_value_mean / H_0_without_mean) * 100
        else:
            protection_pct_mean = 0.0

        results_market.append({
            'county': county,
            'scenario': scenario_label,
            'assessment': 'market',
            'config': HEDONIC_CONFIGS[ACTIVE_CONFIG]['name'],
            'H_0_without_mean': H_0_without_mean,
            'H_0_with_mean': H_0_with_mean,
            'H_0_without_std': np.std(H_0_without_list),
            'H_0_with_std': np.std(H_0_with_list),
            'protection_value_mean': protection_value_mean,
            'protection_pct_mean': protection_pct_mean,
            'xi_h_without': hedonic_damages[county]['without_mangroves'],
            'xi_h_with': hedonic_damages[county]['with_mangroves'],
        })

# Create DataFrame
df_market_results = pd.DataFrame(results_market)

print("\n" + "="*70)
print("MARKET ASSESSMENT RESULTS")
print("="*70)
print(df_market_results[['county', 'scenario', 'H_0_without_mean', 'H_0_with_mean',
                          'protection_value_mean', 'protection_pct_mean']].to_string(index=False))

## Aggregate protection values

In [ ]:
# ============================================================================
# CALCULATE THREE TYPES OF PROTECTION VALUES
# ============================================================================

def calculate_aggregate_protection_new(df_results, df_damages, df_building_values, assessment_type='engineering', stat_type='mean'):
    """
    Calculate three types of protection values:

    1. Per-property: V^m × mean_building_value
    2. Aggregate at-risk: V^m × val_struct_flooded (all flooded properties)
    3. Normalized: Aggregate / total_county_value × 100
    """

    results = []

    for county in COUNTIES:
        for scenario_label in ['baseline', 'climate_change']:

            # Determine scenario codes
            if scenario_label == 'baseline':
                scenario_without = 'S2'  # No mangroves baseline
                scenario_with = 'S1'     # With mangroves baseline
            else:
                scenario_without = 'S2CC'  # No mangroves climate
                scenario_with = 'S1CC'     # With mangroves climate

            if assessment_type == 'engineering':
                result_row = df_results[
                    (df_results['county'] == county) &
                    (df_results['scenario'] == scenario_label)
                ]
                if len(result_row) == 0:
                    continue

                protection_pct = result_row['protection_pct_mean'].values[0] / 100

            else:  # market - UNIFORM across counties
                if scenario_label == 'baseline':
                    protection_pct = df_results[
                        df_results['scenario'] == 'baseline'
                    ]['protection_pct_mean'].iloc[0] / 100
                else:
                    # No climate scenario for market
                    continue

            building_row = df_building_values[
                (df_building_values['county'] == county) &
                (df_building_values['scenario'] == 'S1') &
                (df_building_values['return_period'] == 'RP100')
            ]

            if len(building_row) == 0:
                print(f"Warning: No building data for {county}")
                continue

            mean_building_value = building_row['mean_building_value'].values[0]

            damage_row_without = df_damages[
                (df_damages['county'] == county) &
                (df_damages['scenario'] == scenario_without) &
                (df_damages['return_period'] == 'RP100')
            ]

            if len(damage_row_without) == 0:
                print(f"Warning: No damage data for {county}, {scenario_without}")
                continue

            val_struct_flooded = damage_row_without['val_struct_flooded'].values[0]

            damage_row_with = df_damages[
                (df_damages['county'] == county) &
                (df_damages['scenario'] == scenario_with) &
                (df_damages['return_period'] == 'RP100')
            ]

            if len(damage_row_with) == 0:
                print(f"Warning: No total value data for {county}, {scenario_with}")
                continue

            total_county_value = damage_row_with['val_struct'].values[0]

            # ================================================================
            # STEP 5: Calculate three protection values
            # ================================================================

            # 1. Per-property protection (mean building value)
            per_property_protection = protection_pct * mean_building_value

            # 2. Aggregate at-risk protection (all flooded properties)
            aggregate_at_risk_protection = protection_pct * val_struct_flooded

            # 3. Normalized protection (as % of total county value)
            normalized_protection_pct = (aggregate_at_risk_protection / total_county_value) * 100

            results.append({
                'county': county,
                'scenario': scenario_label,
                'assessment': assessment_type,
                'stat_type': stat_type,
                'protection_pct': protection_pct * 100,  # Store as %

                # Type 1: Per-property
                'mean_building_value': mean_building_value,
                'per_property_protection': per_property_protection,

                # Type 2: Aggregate at-risk
                'val_struct_flooded': val_struct_flooded,
                'aggregate_at_risk_protection': aggregate_at_risk_protection,

                # Type 3: Normalized
                'total_county_value': total_county_value,
                'normalized_protection_pct': normalized_protection_pct
            })

    return pd.DataFrame(results)

In [ ]:
# Clean 'val_struct' to float if it is not already
if df_all_damages['val_struct'].dtype == object:
    df_all_damages['val_struct'] = df_all_damages['val_struct'].astype(str).str.replace('$', '', regex=False).str.replace(',', '', regex=False).astype(float)

# --- MEAN RESULTS ---
df_agg_eng_mean = calculate_aggregate_protection_new(
    df_results_mean, df_all_damages, df, assessment_type='engineering', stat_type='mean'
)

# Market results (Hedonic is constant, but we can label it mean for consistency or do both if applicable)
df_agg_market = calculate_aggregate_protection_new(
    df_market_results, df_all_damages, df, assessment_type='market', stat_type='mean'
)

# Combine all
df_agg_all = pd.concat([df_agg_eng_mean, df_agg_market], ignore_index=True)

# ============================================================================
# Display Results
# ============================================================================

def print_agg_results(df_to_print, title):
    print("\n" + "="*130)
    print(title)
    print("="*130)
    print(f"{'County':<12} {'Scenario':<14} {'Assessment':<12} {'Stat':<8} {'V^m %':<10} "
          f"{'1) Per-Property':<18} {'2) Aggregate At-Risk':<22} {'3) Normalized %':<15}")
    print("-"*130)

    for _, row in df_to_print.iterrows():
        print(f"{row['county']:<12} {row['scenario']:<14} {row['assessment']:<12} {row.get('stat_type', 'mean'):<8} "
              f"{row['protection_pct']:>8.2f}%  "
              f"${row['per_property_protection']:>16,.0f}  "
              f"${row['aggregate_at_risk_protection']:>20,.0f}  "
              f"{row['normalized_protection_pct']:>13.3f}%")

    print("="*130)

# Print Mean
print_agg_results(df_agg_all[df_agg_all['stat_type'] == 'mean'], "PROTECTION VALUES - THREE TYPES (MEAN)")


print("\n" + "="*80)
print("INTERPRETATION GUIDE")
print("="*80)
print("1) Per-Property Protection:")
print("   - What a typical property owner experiences")
print("   - V^m × mean building value")
print("   - Example: '5% protection on $500K property = $25K value'")
print()
print("2) Aggregate At-Risk Protection:")
print("   - Total protection for all properties that would flood WITHOUT mangroves")
print("   - V^m × val_struct_flooded")
print("   - Example: '$2.5B total protection across 10,000 at-risk properties'")
print("   - THIS IS PANEL B (main result)")
print()
print("3) Normalized Protection:")
print("   - Protection as % of total county housing stock")
print("   - (Aggregate at-risk) / (Total county value) × 100")
print("   - Example: '2.5% - mangroves protect value equal to 2.5% of county'")
print("   - THIS IS PANEL C")
print("="*80)

## Export aggregated assessment results

In [ ]:
# Ensure output folder exists
if 'output_folder' not in locals() and 'output_folder' not in globals():
    output_folder = Path('./output')
    output_folder.mkdir(parents=True, exist_ok=True)

# The dataframe df_agg_all is already in a relatively long format
# Let's save it directly
export_filename = 'assessment_results_long.csv'
export_path = output_folder / export_filename

# Select the relevant columns to export based on what was plotted
columns_to_export = [
    'county',
    'scenario',
    'assessment',
    'stat_type',  # 'mean' (kept for downstream compatibility)
    'protection_pct',
    'per_property_protection',
    'aggregate_at_risk_protection',
    'normalized_protection_pct'
]

# Check if df_agg_all exists
if 'df_agg_all' in locals() or 'df_agg_all' in globals():
    df_export = df_agg_all[columns_to_export].copy()

    # Map stat_type to explicit label ('mean_ratio')
    df_export['stat_type'] = df_export['stat_type'] + '_ratio'
    df_export = df_export.rename(columns={'stat_type': 'ratio_type'})

    # Save to CSV
    df_export.to_csv(export_path, index=False)
    print(f"\n✓ Successfully exported results to: {export_path}")

    # Display the first few rows to confirm
    print("\nPreview of exported data:")
    display(df_export.head())
else:
    print("Error: df_agg_all not found. Please run the cell that generates it first.")

## Export Figure 2(b) data

In [ ]:
# Define output folder first
output_folder = Path('./output')

xi_h_data = []
for county in COUNTIES:
    for scenario_label in ['baseline', 'climate_change']:
        # Map to specific data scenario labels
        scenario_with = 'S1' if scenario_label == 'baseline' else 'S1CC'
        scenario_without = 'S2' if scenario_label == 'baseline' else 'S2CC'

        for rp in [10, 25, 50, 100]:
            # Engineering model uses probability adjustment for 'with mangroves' (S1/S1CC)
            xi_h_with = get_damage_ratio(county, rp, scenario_with, df, use_probability=True, df_total=df_total)
            xi_h_without = get_damage_ratio(county, rp, scenario_without, df, use_probability=False, df_total=df_total)

            xi_h_data.append({
                'county': county,
                'climate_scenario': scenario_label,
                'return_period': rp,
                'xi_h_with_mangroves': xi_h_with,
                'xi_h_without_mangroves': xi_h_without
            })

df_fig2b = pd.DataFrame(xi_h_data)
df_fig2b.to_csv(output_folder / 'figure2b_xi_h_engineering.csv', index=False)
print(f"✓ Saved Figure 2(b) data to {output_folder / 'figure2b_xi_h_engineering.csv'}")

# ==================================================================
# For Figure 3: Engineering Hm and Hnom by county and climate scenario
# ==================================================================
# Mean results
df_fig3_mean = df_results_mean[['county', 'scenario', 'H_0_without_mean', 'H_0_with_mean']].copy()
df_fig3_mean = df_fig3_mean.rename(columns={
    'scenario': 'climate_scenario',
    'H_0_without_mean': 'Hnom',
    'H_0_with_mean': 'Hm'
})

df_fig3_mean.to_csv(output_folder / 'figure3_Hm_Hnom_engineering_mean.csv', index=False)
print(f"✓ Saved Figure 3 mean data to {output_folder / 'figure3_Hm_Hnom_engineering_mean.csv'}")

display(df_fig3_mean.head())


## Export $\xi_h$ inputs (Fig 3 data)

In [ ]:
# Ensure output folder exists
output_folder = Path('./output')

# Define the mapping for scenarios
scenarios_mapping = {
    'S1': ('Baseline', 'With_Mangroves'),
    'S2': ('Baseline', 'No_Mangroves'),
    'S1CC': ('Climate', 'With_Mangroves'),
    'S2CC': ('Climate', 'No_Mangroves')
}

for stat_type in ['mean']:
    # Collect the data
    xi_h_inputs_data = []
    for county in COUNTIES:
        for rp in [10, 25, 50, 100]:
            for scenario_code, (clim_scen, mang_state) in scenarios_mapping.items():
                # Calculate unadjusted xi_h
                xi_h_unadjusted = get_damage_ratio(
                    county,
                    rp,
                    scenario_code,
                    df,
                    stat_type=stat_type,
                    use_probability=False,
                    df_total=df_total
                )

                # Calculate probability-adjusted xi_h
                xi_h_adjusted = get_damage_ratio(
                    county,
                    rp,
                    scenario_code,
                    df,
                    stat_type=stat_type,
                    use_probability=True,
                    df_total=df_total
                )

                xi_h_inputs_data.append({
                    'county': county,
                    'return_period': rp,
                    'climate_scenario': clim_scen,
                    'mangrove_state': mang_state,
                    'xi_h': round(xi_h_unadjusted, 4),
                    'xi_h_prob_adjusted': round(xi_h_adjusted, 4)
                })

    # Create DataFrame
    df_xi_h_inputs = pd.DataFrame(xi_h_inputs_data)

    # Ensure correct column order
    df_xi_h_inputs = df_xi_h_inputs[['county', 'return_period', 'climate_scenario', 'mangrove_state', 'xi_h', 'xi_h_prob_adjusted']]

    # Sort the DataFrame as requested
    df_xi_h_inputs = df_xi_h_inputs.sort_values(by=['county', 'climate_scenario', 'mangrove_state', 'return_period'])

    # Save to CSV
    output_file = output_folder / f'xi_h_inputs_{stat_type}.csv'
    df_xi_h_inputs.to_csv(output_file, index=False)
    print(f"✓ Saved {stat_type} xi_h inputs to {output_file}")

    # Display the first few rows to verify
    print(f"\nPreview for {stat_type}:")
    display(df_xi_h_inputs.head(5))

## Market $\xi_h$ table

In [ ]:
import pandas as pd

# Create a list to hold the market damage ratio data
market_xi_data = []

# Market damages are uniform across counties in the current hedonic setup,
# so we can just use the first county as a representative sample.
sample_county = COUNTIES[0]

for rp in [10, 25, 50, 100]:
    # S1 represents 'with mangroves'
    xi_with = get_hedonic_damage_ratio(sample_county, rp, 'S1', hedonic_damages, df, df_total)

    # S2 represents 'without mangroves'
    xi_without = get_hedonic_damage_ratio(sample_county, rp, 'S2', hedonic_damages, df, df_total)

    market_xi_data.append({
        'Return Period (Years)': rp,
        'Scaling Factor': f"{rp/100:.2f}",
        'xi_h (With Mangroves)': xi_with,
        'xi_h (Without Mangroves)': xi_without
    })

df_market_xi = pd.DataFrame(market_xi_data)

print(f"Market (Hedonic) Assessment Damage Ratios")
# print(f"Active Configuration: {HEDONIC_CONFIGS[ACTIVE_CONFIG]['name']}")  # legacy label
print("-" * 60)
display(df_market_xi)